In [177]:
import pandas as pd

In [178]:
data=pd.read_csv('Real estate valuation data set.csv')

In [179]:
data

,No,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
0,1,2012.917,32.0,84.87882,10,24.98298,121.54024,37.9
1,2,2012.917,19.5,306.59470,9,24.98034,121.53951,42.2
2,3,2013.583,13.3,561.98450,5,24.98746,121.54391,47.3
3,4,2013.500,13.3,561.98450,5,24.98746,121.54391,54.8
4,5,2012.833,5.0,390.56840,5,24.97937,121.54245,43.1
...,...,...,...,...,...,...,...,...
409,410,2013.000,13.7,4082.01500,0,24.94155,121.50381,15.4
410,411,2012.667,5.6,90.45606,9,24.97433,121.54310,50.0
411,412,2013.250,18.8,390.96960,7,24.97923,121.53986,40.6
412,413,2013.000,8.1,104.81010,5,24.96674,121.54067,52.5


In [180]:
data=data.drop(columns=['No'])

In [181]:
#separation of features and target
X=data.drop(columns=['Y house price of unit area'])
y=data['Y house price of unit area']

In [182]:
from sklearn.model_selection import train_test_split
X_train,X_val_test,y_train,y_val_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_val,X_test,y_val,y_test=train_test_split(X_val_test,y_val_test,test_size=0.5,random_state=42)

In [183]:
from sklearn.preprocessing import StandardScaler

standard_scaler=StandardScaler()
X_train=standard_scaler.fit_transform(X_train)
X_val=standard_scaler.transform(X_val)
X_test=standard_scaler.transform(X_test)

In [184]:
from sklearn.preprocessing import PolynomialFeatures

degrees=[2,3]
polynomial_features_all=[[X_train,X_val,X_test]]
for degree in degrees:
    polynomial_features=PolynomialFeatures(degree=degree,include_bias=False)
    X_train_poly=polynomial_features.fit_transform(X_train)
    X_val_poly=polynomial_features.transform(X_val)
    X_test_poly=polynomial_features.transform(X_test)
    polynomial_features_all.append([X_train_poly,X_val_poly,X_test_poly])

In [185]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

degree=1
for X_train_poly,X_val_poly,_ in polynomial_features_all:
    linear_regression_model=LinearRegression()
    linear_regression_model.fit(X_train_poly,y_train)
    y_pred=linear_regression_model.predict(X_val_poly)
    print(f'Linear Regression RMSE at degree={degree}: {root_mean_squared_error(y_val,y_pred)}')
    degree+=1

Linear Regression RMSE at degree=1: 8.103324357860481
Linear Regression RMSE at degree=2: 7.009861645420634
Linear Regression RMSE at degree=3: 8.121713064294521


In [186]:
from sklearn.linear_model import Lasso

alphas=[0.01,0.1,1]

for alpha in alphas:
    lasso_model=Lasso(alpha=alpha)
    lasso_model.fit(X_train,y_train)
    y_pred=lasso_model.predict(X_val)
    print(f'Lasso RMSE at alpha={alpha}: {root_mean_squared_error(y_val,y_pred)}')

Lasso RMSE at alpha=0.01: 8.101050154398948
Lasso RMSE at alpha=0.1: 8.085778800637652
Lasso RMSE at alpha=1: 8.17226632229483


In [187]:
best_degree=2
polynomial_features=PolynomialFeatures(degree=best_degree,include_bias=False)
X_train_poly=polynomial_features.fit_transform(X_train)
X_test_poly=polynomial_features.transform(X_test)
linear_regression_model=LinearRegression()
linear_regression_model.fit(X_train_poly,y_train)
y_pred=linear_regression_model.predict(X_test_poly)
print(f'Final test Linear Regression RMSE at degree {best_degree}: {root_mean_squared_error(y_test,y_pred)}')

best_alpha=0.1
lasso_model=Lasso(alpha=best_alpha)
lasso_model.fit(X_train_poly,y_train)
y_pred=lasso_model.predict(X_test_poly)
print(f'Final test Lasso RMSE at alpha {best_alpha}: {root_mean_squared_error(y_test,y_pred)}')

Final test Linear Regression RMSE at degree 2: 5.726988565228985
Final test Lasso RMSE at alpha 0.1: 5.643755892154845
